<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P02/Pratica_02C_atualizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alinhamento de um Par de Sequências de Proteínas Usando Matriz de Substituição

Nesta seção vamos fazer um novo alinhamento. Desta vez, usaremos a matriz de
substituição BLOSUM62 para alinhar as sequências de duas proteínas reais.

Primeiramente, instalaremos o BioPython e definiremos nossa matriz de substituição como a matriz BLOSUM62. Esta matriz ficará definida no objeto *aligner*.

# Alinhamento com sequências reais

**Q4-** Faça uma busca no [Entrez-NCBI](https://www.ncbi.nlm.nih.gov/search/) pela sequência de aminoácidos (formato FASTA) das duas proteínas a seguir:

a. mineralocorticoid receptor de Homo sapiens

b. Receptor de progesterona de Homo sapiens

Qual o código de identificação (accession) para cada uma das sequências? Justifique e discuta a sua escolha frente às opções retornadas na busca (ex: transcript variants, isoformas, sequências preditas vs revisadas).

## Instalando o BioPython

In [1]:
!pip3 install biopython==1.85
import sys, os
from Bio import Align
from Bio.Align import substitution_matrices

matrix = substitution_matrices.load("BLOSUM62")
# print(matrix)
aligner = Align.PairwiseAligner()
aligner.substitution_matrix = matrix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.4 MB/s eta 0:00:00


## Entrada de Sequências

Aqui faremos a entrada com as sequências que queremos alinhar. A matriz de substituição possui um dicionário para lidar com os 20 tipos de aminoácidos. Caracteres outros que apareçam na sequência resultarão em erro.

Cole somente a sequência (ex: `MTALHF...`), excluindo o cabeçalho (linha que começa com '>')

In [6]:
def limpar_fasta(seq):
    """Remove linha de cabeçalho FASTA (iniciada por '>'), quebras de linha
    e espaços, deixando só os resíduos em maiúsculas."""
    linhas = seq.strip().splitlines()
    linhas = [l for l in linhas if not l.startswith('>')]
    return ''.join(linhas).replace(' ', '').upper()


seq1 = "METKGYHSLPEGLDMERRWGQVSQAVERSSLGPTERTDENNYMEIVNVSCVSGAIPNNSTQGSSKEKQEL LPCLQQDNNRPGILTSDIKTELESKELSATVAESMGLYMDSVRDADYSYEQQNQQGSMSPAKIYQNVEQL VKFYKGNGHRPSTLSCVNTPLRSFMSDSGSSVNGGVMRAVVKSPIMCHEKSPSVCSPLNMTSSVCSPAGI NSVSSTTASFGSFPVHSPITQGTPLTCSPNVENRGSRSHSPAHASNVGSPLSSPLSSMKSSISSPPSHCS VKSPVSSPNNVTLRSSVSSPANINNSRCSVSSPSNTNNRSTLSSPAASTVGSICSPVNNAFSYTASGTSA GSSTLRDVVPSPDTQEKGAQEVPFPKTEEVESAISNGVTGQLNIVQYIKPEPDGAFSSSCLGGNSKINSD SSFSVPIKQESTKHSCSGTSFKGNPTVNPFPFMDGSYFSFMDDKDYYSLSGILGPPVPGFDGNCEGSGFP VGIKQEPDDGSYYPEASIPSSAIVGVNSGGQSFHYRIGAQGTISLSRSARDQSFQHLSSFPPVNTLVESW KSHGDLSSRRSDGYPVLEYIPENVSSSTLRSVSTGSSRPSKICLVCGDEASGCHYGVVTCGSCKVFFKRA VEGQHNYLCAGRNDCIIDKIRRKNCPACRLQKCLQAGMNLGARKSKKLGKLKGIHEEQPQQQQPPPPPPP PQSPEEGTTYIAPAKEPSVNTALVPQLSTISRALTPSPVMVLENIEPEIVYAGYDSSKPDTAENLLSTLN RLAGKQMIQVVKWAKVLPGFKNLPLEDQITLIQYSWMCLSSFALSWRSYKHTNSQFLYFAPDLVFNEEKM HQSAMYELCQGMHQISLQFVRLQLTFEEYTIMKVLLLLSTIPKDGLKSQAAFEEMRTNYIKELRKMVTKC PNNSGQSWQRFYQLTKLLDSMHDLVSDLLEFCFYTFRESHALKVEFPAMLVEIISDQLPKVESGNAKPLY FHRK" #@param {type:"string"}
seq2 = "MSRSGCKVGDSSGTAAAHKVLPRGLSPARQLLLPASESPHWSGAPVKPSPQAAAVEVEEEDGSESEESAG PLLKGKPRALGGAAAGGGAAAVPPGAAAGGVALVPKEDSRFSAPRVALVEQDAPMAPGRSPLATTVMDFI HVPILPLNHALLAARTRQLLEDESYDGGAGAASAFAPPRSSPCASSTPVAVGDFPDCAYPPDAEPKDDAY PLYSDFQPPALKIKEEEEGAEASARSPRSYLVAGANPAAFPDFPLGPPPPLPPRATPSRPGEAAVTAAPA SASVSSASSSGSTLECILYKAEGAPPQQGPFAPPPCKAPGASGCLLPRDGLPSTSASAAAAGAAPALYPA LGLNGLPQLGYQAAVLKEGLPQVYPPYLNYLRPDSEASQSPQYSFESLPQKICLICGDEASGCHYGVLTC GSCKVFFKRAMEGQHNYLCAGRNDCIVDKIRRKNCPACRLRKCCQAGMVLGGRKFKKFNKVRVVRALDAV ALPQPVGVPNESQALSQRFTFSPGQDIQLIPPLINLLMSIEPDVIYAGHDNTKPDTSSSLLTSLNQLGER QLLSVVKWSKSLPGFRNLHIDDQITLIQYSWMSLMVFGLGWRSYKHVSGQMLYFAPDLILNEQRMKESSF YSLCLTMWQIPQEFVKLQVSQEEFLCMKVLLLLNTIPLEGLRSQTQFEEMRSSYIRELIKAIGLRQKGVV SSSQRFYQLTKLLDNLHDLVKQLHLYCLNTFIQSRALSVEFPEMMSEVIAAQLPKILAGMVKPLLFHKK" #@param {type:"string"}

seq1 = limpar_fasta(seq1)
seq2 = limpar_fasta(seq2)

print("A sequencia 1 tem %d residuos" % len(seq1))
print("A sequencia 2 tem %d residuos" % len(seq2))

# Validação: verifica se há caracteres fora do alfabeto da matriz de substituição
alfabeto_valido = set(matrix.alphabet)
for nome, seq in [("seq1", seq1), ("seq2", seq2)]:
    invalidos = set(seq) - alfabeto_valido
    if invalidos:
        print(f"AVISO: {nome} tem caracteres fora do alfabeto da BLOSUM62: {sorted(invalidos)}")
        print("Confira se colou a sequência de aminoácidos correta (sem números, sem o '>' do cabeçalho, etc).")

A sequencia 1 tem 984 residuos
A sequencia 2 tem 769 residuos


## Uma função auxiliar para resumir o alinhamento

Para sequências de proteínas reais (que costumam ter centenas de resíduos), o alinhamento impresso inteiro é difícil de interpretar visualmente. Esta função calcula estatísticas básicas — score, % de identidade e número de gaps — que ajudam a responder as perguntas Q4/Q5/Q6 de forma mais objetiva do que só "olhar" o alinhamento.

In [7]:
def resumo_alinhamento(alignment):
    """Imprime score, %identidade e número de gaps de um Bio.Align.Alignment."""
    aligned = alignment.aligned  # blocos (start, end) alinhados em cada sequência
    s1, s2 = alignment[0], alignment[1]

    identicos = sum(1 for a, b in zip(s1, s2) if a == b and a != '-' and b != '-')
    comprimento_alinhado = len(s1)
    gaps = s1.count('-') + s2.count('-')
    pct_identidade = 100 * identicos / comprimento_alinhado if comprimento_alinhado else 0

    print(f"Score: {alignment.score:.1f}")
    print(f"Comprimento do alinhamento: {comprimento_alinhado}")
    print(f"Resíduos idênticos: {identicos} ({pct_identidade:.1f}%)")
    print(f"Total de posições em gap: {gaps}")


def mostrar_trecho(alignment, inicio=0, tamanho=80):
    """Mostra apenas um trecho do alinhamento (útil para sequências longas).
    Use tamanho=None para ver o alinhamento inteiro."""
    texto = str(alignment)
    if tamanho is None:
        print(texto)
        return
    linhas = texto.splitlines()
    for linha in linhas:
        print(linha[inicio:inicio + tamanho])

# Alinhamento Global (padrão)

Primeiro fazemos o alinhamento **global**, cobrindo as duas sequências inteiras. Note que aqui definimos explicitamente uma penalidade de gap (`open_gap_score = -10`, `extend_gap_score = -1.0`).

In [16]:
aligner.mode = 'global'
aligner.open_gap_score = -10
aligner.extend_gap_score = -1

alignments_global = aligner.align(seq1, seq2)
melhor_global = alignments_global[0]

mostrar_trecho(melhor_global, tamanho=80)
print()
resumo_alinhamento(melhor_global)
print("Algoritmo: %s" % aligner.algorithm)

target            0 METKGYHSLPEGLDMERRWGQVSQAVERSSLGPTERTDENNYMEIVNVSCVSGAIPNNST
                  0 |...|-----------------------...|....|-----------.......|----
query             0 MSRSG-----------------------CKVGDSSGT-----------AAAHKVLP----

target           60 QGSSKEKQELLPCLQQDNNRPGILTSDIKTELESKELSATVAESMGLYMDSVRDADYSYE
                 60 .|.|...|.|||.----...|.......|---.|....|.-------------------|
query            22 RGLSPARQLLLPA----SESPHWSGAPVK---PSPQAAAV-------------------E

target          120 QQNQQGSMSPAKIYQNVEQLVKFYKGNGHRPSTLSCVNTPLRSFMSDSGSSVNGGVMRAV
                120 .....||-------...|......||---.|..|--------------|....||...||
query            56 VEEEDGS-------ESEESAGPLLKG---KPRAL--------------GGAAAGGGAAAV

target          180 VKSPIMCHEKSPSVCSPLNMTSSVCSPAGINSVSSTTASFGSFPVHSPITQGTPLTCSPN
                180 ----------.|....-----------.|...|......|-|.|......|..|.-----
query            92 ----------PPGAAA-----------GGVALVPKEDSRF-SAPRVALVEQDAPM-----

target          240 VENR

O alinhamento obtido é razoável? Compare a %identidade obtida com o que se sabe sobre a relação evolutiva entre receptores de hormônios esteroides (mineralocorticoide e progesterona pertencem à mesma superfamília de receptores nucleares).

# Alinhamento Local

**Q5-** Agora repetimos o alinhamento, mas em modo **local**, mantendo os mesmos parâmetros de gap. Que diferenças você observa em relação ao alinhamento global (score, %identidade, comprimento do alinhamento, região das sequências coberta)? O que isso nos diz sobre as duas proteínas?

In [13]:
aligner.mode = 'local'
aligner.open_gap_score = -10
aligner.extend_gap_score = -1

alignments_local = aligner.align(seq1, seq2)
melhor_local = alignments_local[0]

mostrar_trecho(melhor_local, tamanho=80)
print()
resumo_alinhamento(melhor_local)
print("Algoritmo: %s" % aligner.algorithm)

target          574 PVLEYI-PENVSSSTLRSVSTGSSRPSKICLVCGDEASGCHYGVVTCGSCKVFFKRAVEG
                  0 |.|.|.-|....|....--....|.|.||||.||||||||||||.||||||||||||.||
query           375 PYLNYLRPDSEASQSPQ--YSFESLPQKICLICGDEASGCHYGVLTCGSCKVFFKRAMEG

target          633 QHNYLCAGRNDCIIDKIRRKNCPACRLQKCLQAGMNLGARKSKKLGKLKGIHE-EQPQQQ
                 60 |||||||||||||.|||||||||||||.||.||||.||.||.||..|......-......
query           433 QHNYLCAGRNDCIVDKIRRKNCPACRLRKCCQAGMVLGGRKFKKFNKVRVVRALDAVALP

target          692 QPPPPPPPPQSPEEGTTYIAPAKEPSVNTALVPQLSTISRALTPSPVMVLENIEPEIVYA
                120 ||...|...|......|.-----.|.....|.|.|-----------...|..|||...||
query           493 QPVGVPNESQALSQRFTF-----SPGQDIQLIPPL-----------INLLMSIEPDVIYA

target          752 GYDSSKPDTAENLLSTLNRLAGKQMIQVVKWAKVLPGFKNLPLEDQITLIQYSWMCLSSF
                180 |.|..||||...||..||.|...|...||||.|.||||.||...|||||||||||.|..|
query           537 GHDNTKPDTSSSLLTSLNQLGERQLLSVVKWSKSLPGFRNLHIDDQITLIQYSWMSLMVF

target          812 ALSW

# Efeito da Penalidade de Gap

**Q6-** Aqui, repetimos o alinhamento global, mas aumentamos a penalidade de abertura de gap de −10 para **−15**. O que mudou no alinhamento obtido? O que podemos concluir sobre o alinhamento das duas proteínas? O que as diferenças no código nos ensinam sobre o alinhamento de sequências?

Fique à vontade para editar o valor de `open_gap_score` na célula abaixo e testar outros valores.

In [14]:
aligner.mode = 'global'
aligner.open_gap_score = -15
aligner.extend_gap_score = -0.5

alignments_gap15 = aligner.align(seq1, seq2)
melhor_gap15 = alignments_gap15[0]

mostrar_trecho(melhor_gap15, tamanho=80)
print()
resumo_alinhamento(melhor_gap15)
print("Algoritmo: %s" % aligner.algorithm)

target            0 METKG------------YHSLPEGLDMERRWGQVSQAVERSSLGPTERTDENNYMEIVNV
                  0 |...|------------...||.||...|............|..|..........|.---
query             0 MSRSGCKVGDSSGTAAAHKVLPRGLSPARQLLLPASESPHWSGAPVKPSPQAAAVEV---

target           48 SCVSGAIPNNSTQGSSKEKQELLPCLQQDNNR--------------PGILTSDIKTELES
                 60 ----------.....|.......|.|......--------------||......--.|..
query            57 ----------EEEDGSESEESAGPLLKGKPRALGGAAAGGGAAAVPPGAAAGGV--ALVP

target           94 KELSATVAESMGLYMDSVRDADYSYEQQNQQGSMSPAKIYQNVEQLVKFYKGNGHRPSTL
                120 ||.|...|....|.--------------.|...|.|..----------------------
query           105 KEDSRFSAPRVALV--------------EQDAPMAPGR----------------------

target          154 SCVNTPLRSFMSDSGSSVNGGVMRAVVKSPIMCHEKSPSVCSPLNMTSSVCSPAGINSVS
                180 |...|....|........|.....|......----...|........|....|......|
query           129 SPLATTVMDFIHVPILPLNHALLAARTRQLL----EDESYDGGAGAASAFAPPRSSPCAS

target          214 STTA

## Comparando os três alinhamentos lado a lado

Para facilitar a discussão da Q6, esta célula resume score, %identidade e gaps dos três alinhamentos feitos acima (global com gap −10, local com gap −10, global com gap −15).

In [15]:
import pandas as pd

def stats_dict(alignment, nome):
    s1, s2 = alignment[0], alignment[1]
    identicos = sum(1 for a, b in zip(s1, s2) if a == b and a != '-' and b != '-')
    comprimento = len(s1)
    gaps = s1.count('-') + s2.count('-')
    return {
        "Alinhamento": nome,
        "Score": round(alignment.score, 1),
        "Comprimento": comprimento,
        "% Identidade": round(100 * identicos / comprimento, 1) if comprimento else 0,
        "Gaps": gaps,
    }

resumo_df = pd.DataFrame([
    stats_dict(melhor_global, "Global (gap open=-10)"),
    stats_dict(melhor_local, "Local (gap open=-10)"),
    stats_dict(melhor_gap15, "Global (gap open=-15)"),
])
resumo_df

,Alinhamento,Score,Comprimento,% Identidade,Gaps
0,Global (gap open=-10),1141.0,1048,30.6,343
1,Local (gap open=-10),1147.0,412,54.1,20
2,Global (gap open=-15),1012.5,1011,29.6,269


## Síntese

Com base na tabela acima e nos alinhamentos, responda:
1. As duas proteínas são homólogas? O que sustenta essa conclusão (%identidade, cobertura do alinhamento)?
2. Por que o alinhamento local tende a ter %identidade maior que o global, quando as sequências têm regiões muito divergentes fora do domínio conservado?
3. Aumentar a penalidade de abertura de gap tende a produzir *mais* ou *menos* gaps no alinhamento final? Isso é sempre desejável?